# HPCGuard – HPC Anomaly Detection
Clean-only anomaly detection baseline for the three supplied hardware performance counters. An anomaly is not proof of a backdoor.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib


## 1. Load and inspect the clean dataset
Upload `trace.csv` to the Colab session, or change `CSV_PATH` to its Drive path.

In [ ]:
CSV_PATH = '/content/trace.csv'
df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
display(df.head())
display(df.dtypes)
print('Missing values:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
display(df.describe().T)


## 2. Train and calibrate Isolation Forest
Split clean-only data 80:20. Fit scaling on training rows only. The 95th percentile threshold targets a 5% clean calibration alert rate, not 95% accuracy.

In [ ]:
FEATURES = ['cache-references', 'cycles', 'LLC-loads']
X = df[FEATURES].apply(pd.to_numeric, errors='coerce')
if X.isna().any().any() or not np.isfinite(X.to_numpy()).all(): raise ValueError('Invalid HPC values')
X_train, X_cal = train_test_split(X, test_size=0.20, random_state=42)
model = Pipeline([('scaler', StandardScaler()), ('iforest', IsolationForest(n_estimators=300, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1))])
model.fit(X_train)
scores = -model.decision_function(X_cal)
threshold = float(np.percentile(scores, 95))
pred = (scores > threshold).astype(int)
print('Train/calibration:', len(X_train), len(X_cal))
print('Threshold:', threshold, 'alerts:', int(pred.sum()))


In [ ]:
plt.hist(scores, bins=25, edgecolor='black')
plt.axvline(threshold, linestyle='--', label='95th percentile threshold')
plt.xlabel('Anomaly score (higher = more anomalous)'); plt.ylabel('Clean calibration traces')
plt.legend(); plt.show()


## 3. Save detector artifact
The artifact stores the fitted pipeline, feature order, threshold and metadata.

In [ ]:
artifact = {'project_name':'HPCGuard', 'model_name':'IsolationForest', 'model':model, 'features':FEATURES, 'threshold':threshold, 'calibration_percentile':95, 'random_state':42}
joblib.dump(artifact, 'hpcguard_final.pkl')
print('Saved hpcguard_final.pkl')


## Limitations
No backdoor-labelled training traces or private validation labels are present in this dataset. Accuracy, TPR, FPR, F1 and AUROC must be calculated on the organizers' labelled validation set. The 5% clean calibration alert rate is not a detection metric.